# Horse Handicapping Model
### Version: 1.9
### Date of Last Edits: Sept 21, 2026
### Created by: John Little

In [19]:
import io
import numpy as np
import pandas as pd

# Define expected column schemas for automatic fallback
RUNNER_HEADERS = [
    "PROGRAM",
    "HORSE_NAME",
    "POST",
    "ML_ODDS",
    "LIVE_ODDS",
    "RUN_STYLE",
    "RUN_STYLE_PTS",
    "PRM_PWR",
    "AVG_SPD",
    "BACK_SPD",
    "SPD_LR",
    "AVG_CLS",
    "LAST_CLS",
    "AVG_DIST_SPD",
    "BEST_SPD",
    "W_JKY",
    "W_TRN",
    "E1",
    "E2",
    "LP",
    "DAYS_OFF",
]

STATS_HEADERS = [
    "BREED",
    "TRACK",
    "RACE_NUMBER",
    "STAT_SET",
    "RACES",
    "SPEED_BIAS",
    "IV_RAIL",
    "IV_1TO3",
    "IV_4TO7",
    "IV_8PLUS",
    "IV_E",
    "IV_EP",
    "IV_P",
    "IV_S",
]


def parse_odds_to_decimal(odds_val):
  """Converts fractional odds strings (e.g., '7/2', '5/2', '20') or floats into

  decimal values for comparison. Returns np.nan on failure.
  """
  if pd.isna(odds_val):
    return np.nan
  s = str(odds_val).strip()
  if not s or s.upper() == "NAN":
    return np.nan
  try:
    if "/" in s:
      num, den = s.split("/")
      return float(num) / float(den)
    return float(s)
  except Exception:
    return np.nan


def safe_float(val, default=0.0):
  """Safely converts any value to float, defaulting on failure."""
  try:
    if pd.isna(val):
      return default
    return float(val)
  except Exception:
    return default


def ingest_csv_clipboard(title_msg, expected_headers=None):
  """Ingests CSV data directly from system clipboard using pandas.

  Automatically handles inputs with or without header rows.
  """
  print("=" * 60)
  print(title_msg)
  print("=" * 60)

  try:
    # Read clipboard as raw table with no assumed headers
    df_raw = pd.read_clipboard(sep=",", header=None)
  except Exception as e:
    print(f"Error reading clipboard: {e}")
    return None

  if df_raw is None or df_raw.empty:
    print("Error: Clipboard is empty or could not be parsed.")
    return None

  # Check if Row 0 contains known header keywords
  row0_str = " ".join([str(x).upper() for x in df_raw.iloc[0].values])
  header_keywords = [
      "BREED",
      "PROGRAM",
      "HORSE_NAME",
      "STAT_SET",
      "RUN_STYLE",
      "POST",
  ]
  has_headers = any(kw in row0_str for kw in header_keywords)

  if has_headers:
    # Use Row 0 as headers and slice remaining rows
    df = df_raw.iloc[1:].copy()
    df.columns = [str(x).strip().upper() for x in df_raw.iloc[0].values]
  else:
    # Row 0 is data; assign expected headers if column count matches
    df = df_raw.copy()
    if expected_headers and len(df.columns) == len(expected_headers):
      df.columns = [h.upper() for h in expected_headers]
      print(
          "[Auto-Fix] Header-less CSV detected. Applied default schema headers."
      )
    else:
      df.columns = [f"COL_{i}" for i in range(len(df.columns))]

  return df


if __name__ == "__main__":
  # --------------------------------------------------------------------
  # 1. MAIN RUNNER DATA INGESTION & CLEANING
  # --------------------------------------------------------------------
  input(
      "PAUSE 1: Copy your RUNNER CSV to clipboard, then press Enter..."
  )
  df_race = ingest_csv_clipboard(
      "INGESTING MAIN RUNNER CSV FROM CLIPBOARD", expected_headers=RUNNER_HEADERS
  )

  if df_race is not None:
    primary_key = "POST" if "POST" in df_race.columns else "PROGRAM"
    if primary_key not in df_race.columns:
      primary_key = df_race.columns[0]

    df_race[primary_key] = pd.to_numeric(df_race[primary_key], errors="coerce")
    df_race = df_race.dropna(subset=[primary_key])
    df_race[primary_key] = df_race[primary_key].astype("Int64")
    df_race.set_index(primary_key, inplace=False, drop=False)

    # Clean & Format Odds Fields
    if "ML_ODDS" in df_race.columns:
      df_race["ML_ODDS_DEC"] = df_race["ML_ODDS"].apply(parse_odds_to_decimal)
    else:
      df_race["ML_ODDS_DEC"] = np.nan

    if "LIVE_ODDS" in df_race.columns:
      df_race["LIVE_ODDS_DEC"] = df_race["LIVE_ODDS"].apply(
          parse_odds_to_decimal
      )
      df_race["LIVE_ODDS_DEC"] = df_race["LIVE_ODDS_DEC"].fillna(
          df_race["ML_ODDS_DEC"]
      )
    else:
      df_race["LIVE_ODDS_DEC"] = df_race["ML_ODDS_DEC"]

    # Clean & Format Run Style Fields
    if "RUN_STYLE" in df_race.columns:
      df_race["RUN_STYLE"] = (
          df_race["RUN_STYLE"].astype(str).str.strip().str.upper()
      )
    else:
      df_race["RUN_STYLE"] = "NA"

    if "RUN_STYLE_PTS" in df_race.columns:
      df_race["RUN_STYLE_PTS"] = (
          pd.to_numeric(df_race["RUN_STYLE_PTS"], errors="coerce")
          .fillna(0)
          .astype(int)
      )
    else:
      df_race["RUN_STYLE_PTS"] = 0

    print(
        f"\nSuccessfully ingested {len(df_race)} horses across"
        f" {len(df_race.columns)} columns!"
    )

  # --------------------------------------------------------------------
  # 2. RACE STATS DATA INGESTION & BIAS BLENDING
  # --------------------------------------------------------------------
  print("\n" + "=" * 60)
  input(
      "PAUSE 2: Copy your RACE STATS CSV to clipboard, then press Enter..."
  )
  df_stats = ingest_csv_clipboard(
      "INGESTING RACE STATS CSV FROM CLIPBOARD", expected_headers=STATS_HEADERS
  )

  clean_breed = "Thoroughbred"
  clean_track = "Unknown Track"
  race_num = "1"
  active_speed_bias = 0.0
  post_iv_mapping = {}
  run_style_iv_mapping = {"E": 1.0, "E/P": 1.0, "P": 1.0, "S": 1.0}

  if df_stats is not None:
    if "BREED" in df_stats.columns:
      clean_breed = str(df_stats["BREED"].iloc[0]).strip().title()
    if "TRACK" in df_stats.columns:
      clean_track = str(df_stats["TRACK"].iloc[0]).strip().title()
    if "RACE_NUMBER" in df_stats.columns:
      race_num = str(df_stats["RACE_NUMBER"].iloc[0]).strip()
    elif "RACE_NUM" in df_stats.columns:
      race_num = str(df_stats["RACE_NUM"].iloc[0]).strip()

    df_stats["STAT_SET"] = df_stats["STAT_SET"].astype(str).str.strip().str.title()

    week_rows = df_stats[df_stats["STAT_SET"] == "Week"]
    meet_rows = df_stats[df_stats["STAT_SET"] == "Meet"]

    if not week_rows.empty and not meet_rows.empty:
      week_data = week_rows.iloc[0]
      meet_data = meet_rows.iloc[0]
      week_races = safe_float(week_data["RACES"], 0.0)

      # Dynamic Blending Split Based on Sample Size (<15 vs >=15)
      if week_races < 15:
        w_week, w_meet = 0.00, 1.00
        blend_msg = (
            "Using 100% Meet stats (Week sample too small:"
            f" {int(week_races)} < 15 races)"
        )
      else:
        w_week, w_meet = 0.65, 0.35
        blend_msg = (
            "Blending 65% Week / 35% Meet (Week sample sufficient:"
            f" {int(week_races)} >= 15 races)"
        )

      print(f"\n[Rule Applied] {blend_msg}")

      active_speed_bias = (w_week * safe_float(week_data["SPEED_BIAS"])) + (
          w_meet * safe_float(meet_data["SPEED_BIAS"])
      )

      # Post Position Impact Values
      post_iv_mapping = {
          "RAIL": round(
              (w_week * safe_float(week_data["IV_RAIL"]))
              + (w_meet * safe_float(meet_data["IV_RAIL"])),
              2,
          ),
          "1-3": round(
              (w_week * safe_float(week_data["IV_1TO3"]))
              + (w_meet * safe_float(meet_data["IV_1TO3"])),
              2,
          ),
          "4-7": round(
              (w_week * safe_float(week_data["IV_4TO7"]))
              + (w_meet * safe_float(meet_data["IV_4TO7"])),
              2,
          ),
          "8+": round(
              (w_week * safe_float(week_data["IV_8PLUS"]))
              + (w_meet * safe_float(meet_data["IV_8PLUS"])),
              2,
          ),
      }

      # Run Style Impact Values
      run_style_iv_mapping = {
          "E": round(
              (w_week * safe_float(week_data.get("IV_E"), 1.0))
              + (w_meet * safe_float(meet_data.get("IV_E"), 1.0)),
              2,
          ),
          "E/P": round(
              (w_week * safe_float(week_data.get("IV_EP"), 1.0))
              + (w_meet * safe_float(meet_data.get("IV_EP"), 1.0)),
              2,
          ),
          "P": round(
              (w_week * safe_float(week_data.get("IV_P"), 1.0))
              + (w_meet * safe_float(meet_data.get("IV_P"), 1.0)),
              2,
          ),
          "S": round(
              (w_week * safe_float(week_data.get("IV_S"), 1.0))
              + (w_meet * safe_float(meet_data.get("IV_S"), 1.0)),
              2,
          ),
      }

    elif not meet_rows.empty:
      meet_data = meet_rows.iloc[0]
      print(
          "\n[Rule Applied] Week data missing. Defaulting to 100% Meet stats."
      )
      active_speed_bias = safe_float(meet_data["SPEED_BIAS"])

      post_iv_mapping = {
          "RAIL": round(safe_float(meet_data["IV_RAIL"]), 2),
          "1-3": round(safe_float(meet_data["IV_1TO3"]), 2),
          "4-7": round(safe_float(meet_data["IV_4TO7"]), 2),
          "8+": round(safe_float(meet_data["IV_8PLUS"]), 2),
      }

      run_style_iv_mapping = {
          "E": round(safe_float(meet_data.get("IV_E"), 1.0), 2),
          "E/P": round(safe_float(meet_data.get("IV_EP"), 1.0), 2),
          "P": round(safe_float(meet_data.get("IV_P"), 1.0), 2),
          "S": round(safe_float(meet_data.get("IV_S"), 1.0), 2),
      }
    else:
      print("Error: Could not locate Meet stats in the scraped table.")

    print(f"Active Blended Speed Bias: {active_speed_bias:.3f}")
    print(f"Blended Post Impact Value Map: {post_iv_mapping}")
    print(f"Blended Run Style Impact Value Map: {run_style_iv_mapping}")
#-------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------
#-------------------------------------------------------------------------------------------------------------------------------------------------
#COMBINED BLOCKS 2-5 TO RUN IN A SINGLE BLOCK
# BLOCK 2: Weight Setup & Scaling (based on race stats)

# Master weight definitions (Scaled to 100% Base Weights)
weights_dict = {
    'Thoroughbred': {
        'W_Speed': 0.16,
        'W_Power': 0.09,
        'W_Class': 0.14,
        'W_Distance': 0.11,
        'W_Driver': 0.06,
        'W_Trainer': 0.06,
        'W_Early': 0.15,
        'W_Finish': 0.10,
        'W_Recency': 0.03,
        'W_Course': 0.02,
        'W_Market': 0.08,
        'W_PostBias': 0.00,
        'W_Style': 0.00,
    },
    'Harness': {
        'W_Speed': 0.13,
        'W_Power': 0.07,
        'W_Class': 0.11,
        'W_Distance': 0.04,
        'W_Driver': 0.18,
        'W_Trainer': 0.06,
        'W_Early': 0.18,
        'W_Finish': 0.08,
        'W_Recency': 0.04,
        'W_Course': 0.03,
        'W_Market': 0.08,
        'W_PostBias': 0.00,
        'W_Style': 0.00,
    },
    'Quarter Horse': {
        'W_Speed': 0.22,
        'W_Power': 0.10,
        'W_Class': 0.09,
        'W_Distance': 0.02,
        'W_Driver': 0.07,
        'W_Trainer': 0.07,
        'W_Early': 0.30,
        'W_Finish': 0.00,
        'W_Recency': 0.02,
        'W_Course': 0.00,
        'W_Market': 0.11,
        'W_PostBias': 0.00,
        'W_Style': 0.00,
    },
    'Churchill Downs': {
        'W_Speed': 0.16,
        'W_Power': 0.09,
        'W_Class': 0.17,
        'W_Distance': 0.11,
        'W_Driver': 0.08,
        'W_Trainer': 0.08,
        'W_Early': 0.11,
        'W_Finish': 0.12,
        'W_Recency': 0.03,
        'W_Course': 0.02,
        'W_Market': 0.03,
        'W_PostBias': 0.00,
        'W_Style': 0.00,
    },
}

weights_df = pd.DataFrame(weights_dict)

# 1. Select Base Weights
if clean_track == 'Churchill Downs':
  active_weights = weights_df['Churchill Downs'].copy()
  weight_mode_msg = 'SPECIAL CHURCHILL DOWNS WEIGHTS'
else:
  if clean_breed in weights_df.columns:
    active_weights = weights_df[clean_breed].copy()
    weight_mode_msg = f'Standard {clean_breed} Base Weights'
  else:
    raise ValueError(f"Breed '{clean_breed}' not found in model weights.")

# 2. Detect Meet Sample Size
meet_var_candidates = [
    'meet_races',
    'meet_race_count',
    'races_meet',
    'MEET_RACES',
    'races_in_meet',
    'meet_count',
]
active_meet_count = 10  # Explicit default for low-sample meets unless overridden

for var in meet_var_candidates:
  if var in locals():
    active_meet_count = locals()[var]
    break

# Set blending weights based on 3-tier sample size rule
if active_meet_count < 6:
  meet_weight = 0.00
  base_weight = 1.00
  sample_scale_factor = 0.00
elif active_meet_count <= 15:
  meet_weight = 0.10
  base_weight = 0.90
  sample_scale_factor = 0.10
else:
  meet_weight = 0.65
  base_weight = 0.35
  sample_scale_factor = 1.00

# 3. Dynamic Early Pace Scaling
bias_decimal = (
    active_speed_bias / 100.0 if active_speed_bias > 1.0 else active_speed_bias
)
base_w_early = active_weights['W_Early']

if bias_decimal > 0.55:
  raw_early_boost = (bias_decimal - 0.55) * 0.5
  early_pace_boost = raw_early_boost * sample_scale_factor
  active_weights['W_Early'] = base_w_early + early_pace_boost

  early_msg = (
      f"Boosted from {base_w_early:.1%} to {active_weights['W_Early']:.1%} "
      f'(Dampened to {sample_scale_factor:.0%} weight | Meet Races:'
      f' {active_meet_count})'
  )
else:
  early_msg = (
      f'Kept at Base {base_w_early:.1%} (Speed Bias: {bias_decimal:.1%} <= 55%)'
  )

# Unpack scalar variables
w_speed = active_weights['W_Speed']
w_power = active_weights['W_Power']
w_class = active_weights['W_Class']
w_distance = active_weights['W_Distance']
w_driver = active_weights['W_Driver']
w_trainer = active_weights['W_Trainer']
w_early = active_weights['W_Early']
w_finish = active_weights['W_Finish']
w_recency = active_weights['W_Recency']
w_course = active_weights['W_Course']
w_market = active_weights['W_Market']
w_postbias = active_weights['W_PostBias']
w_style = active_weights['W_Style']

import sys
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------------------------------------------------------------
# BLOCK 3: Dynamic Post Bias Engine & Confirmation

print("\n" + "=" * 60)
print("CALCULATING DYNAMIC POST & STYLE BIAS BONUSES")
print("=" * 60)

# Fallback defaults for variables if not defined globally
clean_breed = clean_breed if "clean_breed" in locals() else "Thoroughbred"
post_iv_mapping = (
    post_iv_mapping
    if "post_iv_mapping" in locals()
    else {"RAIL": 1.00, "1-3": 1.00, "4-7": 1.00, "8+": 1.00}
)
run_style_iv_mapping = (
    run_style_iv_mapping
    if "run_style_iv_mapping" in locals()
    else {"E": 1.00, "E/P": 1.00, "P": 1.00, "S": 1.00}
)

# 1. Multiplier Setup by Breed
if clean_breed == "Harness":
  post_iv_multiplier = 7.0
  style_iv_multiplier = 7.0
elif clean_breed == "Quarter Horse":
  post_iv_multiplier = 4.0
  style_iv_multiplier = 5.0
else:  # Thoroughbred
  post_iv_multiplier = 6.0
  style_iv_multiplier = 6.0

MAX_BIAS_CAP = 5.00


def get_post_category(post_val):
  """Maps numeric post position to post position category string."""
  try:
    p = int(float(post_val))
    if p == 1:
      return "RAIL"
    elif 1 <= p <= 3:
      return "1-3"
    elif 4 <= p <= 7:
      return "4-7"
    elif p >= 8:
      return "8+"
  except (ValueError, TypeError):
    pass
  return "1-3"


def calculate_post_bonus(post_cat, iv_map, multiplier, max_cap=MAX_BIAS_CAP):
  iv = iv_map.get(post_cat, 1.00)
  if iv <= 0:  # Safety guard for missing track data
    iv = 1.00

  iv_diff = max(0.0, iv - 1.00)  # Positive bonus ONLY (No negative penalties)

  if iv_diff >= 0.03:
    raw_bonus = iv_diff * multiplier
    return round(min(max_cap, raw_bonus), 2)
  return 0.0


def calculate_style_bonus(
    style_code, style_pts, iv_map, multiplier, max_cap=MAX_BIAS_CAP
):
  code = str(style_code).strip().upper()
  iv = iv_map.get(code, 1.00)
  if iv <= 0:  # Safety guard for missing track data
    iv = 1.00

  iv_diff = max(0.0, iv - 1.00)  # Positive bonus ONLY (No negative penalties)

  if iv_diff >= 0.03:
    try:
      pts_factor = min(max(float(style_pts) / 8.0, 0.0), 1.0)
    except (ValueError, TypeError):
      pts_factor = 0.50

    raw_bonus = iv_diff * multiplier * pts_factor
    return round(min(max_cap, raw_bonus), 2)
  return 0.0


# Extract active meet race count directly from df_stats if available
if "df_stats" in locals() and df_stats is not None:
  meet_rows = df_stats[
      df_stats["STAT_SET"].astype(str).str.strip().str.title() == "Meet"
  ]
  if not meet_rows.empty and "RACES" in meet_rows.columns:
    try:
      active_meet_count = float(meet_rows.iloc[0]["RACES"])
    except Exception:
      active_meet_count = 0.0
  elif "active_meet_count" not in locals():
    active_meet_count = 100.0  # Default bypass if df_stats is unstructured
else:
  if "active_meet_count" not in locals():
    active_meet_count = 100.0

# 2. Apply Bonuses to DataFrame
if "df_race" in locals() and df_race is not None:

  # SAMPLE SIZE GUARD: Zero out bonuses if meet sample is under 15 races
  if active_meet_count < 15:
    print(
        f"\n[NOTICE] Low Meet Sample Size ({int(active_meet_count)} races < 15"
        " threshold). All Post & Style bonuses set to 0.00."
    )
    df_race["POST_CAT"] = "N/A"
    df_race["POST_IV"] = 1.00
    df_race["POST_BONUS"] = 0.00
    df_race["STYLE_IV"] = 1.00
    df_race["STYLE_BONUS"] = 0.00
  else:
    primary_post_col = "POST" if "POST" in df_race.columns else "PROGRAM"

    # Apply Post Position Bonuses
    df_race["POST_CAT"] = df_race[primary_post_col].apply(get_post_category)
    df_race["POST_IV"] = df_race["POST_CAT"].apply(
        lambda cat: (
            1.00
            if post_iv_mapping.get(cat, 1.00) <= 0
            else post_iv_mapping.get(cat, 1.00)
        )
    )
    df_race["POST_BONUS"] = df_race["POST_CAT"].apply(
        lambda cat: calculate_post_bonus(
            cat, post_iv_mapping, post_iv_multiplier
        )
    )

    # Apply Running Style Bonuses
    if "RUN_STYLE" in df_race.columns:
      df_race["STYLE_IV"] = df_race["RUN_STYLE"].apply(
          lambda style: (
              1.00
              if run_style_iv_mapping.get(style, 1.00) <= 0
              else run_style_iv_mapping.get(style, 1.00)
          )
      )
      df_race["STYLE_BONUS"] = df_race.apply(
          lambda row: calculate_style_bonus(
              row.get("RUN_STYLE", "NA"),
              row.get("RUN_STYLE_PTS", 0),
              run_style_iv_mapping,
              style_iv_multiplier,
          ),
          axis=1,
      )
    else:
      df_race["STYLE_IV"] = 1.00
      df_race["STYLE_BONUS"] = 0.00

# -------------------------------------------------------------------------------------------------------------------------------
## BLOCK 4: Composite Sub-Score & Weighted Final Score Engine

df_calc = df_race.copy()

if "LIVE_ODDS" in df_calc.columns:
  df_calc = df_calc[
      ~df_calc["LIVE_ODDS"].astype(str).str.upper().str.contains("SCR", na=False)
  ]

if "STATUS" in df_calc.columns:
  df_calc = df_calc[
      ~df_calc["STATUS"].astype(str).str.upper().str.contains("SCR", na=False)
  ]

df_calc = df_calc.reset_index(drop=True)

# 0. MISSING DATA IMPUTATION
if "LIVE_ODDS_DEC" in df_calc.columns:
  df_calc["LIVE_ODDS_DEC"] = (
      pd.to_numeric(df_calc["LIVE_ODDS_DEC"], errors="coerce")
      .replace(0, np.nan)
      .fillna(df_calc.get("ML_ODDS_DEC", np.nan))
  )

impute_num_cols = [
    "PRM_PWR",
    "AVG_SPD",
    "BACK_SPD",
    "SPD_LR",
    "AVG_CLS",
    "LAST_CLS",
    "AVG_DIST_SPD",
    "BEST_SPD",
    "W_JKY",
    "W_TRN",
    "E1",
    "E2",
    "LP",
    "DAYS_OFF",
]

for col in impute_num_cols:
  if col in df_calc.columns:
    df_calc[col] = pd.to_numeric(df_calc[col], errors="coerce")
    valid_entries = df_calc[col].notna() & (df_calc[col] > 0)
    if valid_entries.any():
      field_avg = round(df_calc.loc[valid_entries, col].mean(), 1)
      df_calc[col] = df_calc[col].apply(
          lambda x: field_avg if pd.isna(x) or x <= 0 else x
      )
    else:
      df_calc[col] = df_calc[col].fillna(0.0)


# 1. CALCULATE RAW COMPOSITE SUB-SCORES (Matching JS Fallback Logic)
def calc_comp_speed(row):
  avg_dist = row.get("AVG_DIST_SPD", 0)
  spd_lr = row.get("SPD_LR", 0)
  avg_spd = row.get("AVG_SPD", 0)
  if avg_dist > 0 and spd_lr > 0:
    return (0.70 * avg_dist) + (0.30 * spd_lr)
  elif spd_lr > 0:
    return spd_lr
  elif avg_dist > 0:
    return avg_dist
  else:
    return avg_spd


df_calc["COMP_SPEED"] = df_calc.apply(calc_comp_speed, axis=1)
df_calc["COMP_POWER"] = df_calc.get("PRM_PWR", 0)

# Class Fallback: AVG_CLS > 0 -> LAST_CLS -> 0
df_calc["COMP_CLASS"] = df_calc.apply(
    lambda r: (
        r["AVG_CLS"]
        if r.get("AVG_CLS", 0) > 0
        else (r["LAST_CLS"] if r.get("LAST_CLS", 0) > 0 else 0)
    ),
    axis=1,
)

# Distance Fallback: AVG_DIST_SPD > 0 -> BEST_SPD / BACK_SPD -> 0
df_calc["COMP_DISTANCE"] = df_calc.apply(
    lambda r: (
        r["AVG_DIST_SPD"]
        if r.get("AVG_DIST_SPD", 0) > 0
        else (
            r["BEST_SPD"]
            if r.get("BEST_SPD", 0) > 0
            else r.get("BACK_SPD", 0)
        )
    ),
    axis=1,
)

df_calc["COMP_DRIVER"] = df_calc.get("W_JKY", 0)
df_calc["COMP_TRAINER"] = df_calc.get("W_TRN", 0)


def calc_comp_early(row):
  e1 = row.get("E1", 0)
  e2 = row.get("E2", 0)
  if e1 > 0 and e2 > 0:
    return (0.50 * e1) + (0.50 * e2)
  elif e2 > 0:
    return e2
  else:
    return e1


df_calc["COMP_EARLY"] = df_calc.apply(calc_comp_early, axis=1)
df_calc["COMP_FINISH"] = df_calc.get("LP", 0)


def score_recency(days):
  try:
    d = float(days)
    if 14 <= d <= 45:
      return 100.0
    elif d < 14:
      return 85.0
    elif 46 <= d <= 90:
      return 70.0
    else:
      return 50.0
  except (ValueError, TypeError):
    return 75.0


df_calc["COMP_RECENCY"] = df_calc["DAYS_OFF"].apply(score_recency)
df_calc["COMP_COURSE"] = df_calc.get("TRACK_WIN_PCT", 0.0)

if "LIVE_ODDS_DEC" in df_calc.columns:
  df_calc["COMP_MARKET"] = df_calc["LIVE_ODDS_DEC"].apply(
      lambda odds: (1.0 / (odds + 1.0)) * 100.0 if odds > 0 else 0.0
  )
elif "ML_ODDS_DEC" in df_calc.columns:
  df_calc["COMP_MARKET"] = df_calc["ML_ODDS_DEC"].apply(
      lambda odds: (1.0 / (odds + 1.0)) * 100.0 if odds > 0 else 0.0
  )
else:
  df_calc["COMP_MARKET"] = 0.0

# 2. APPLY WEIGHTS (V16 Alignment: Cap W_Market at 3%)
w_market = 0.03

df_calc["SCORE_SPEED"] = df_calc["COMP_SPEED"] * w_speed
df_calc["SCORE_POWER"] = df_calc["COMP_POWER"] * w_power
df_calc["SCORE_CLASS"] = df_calc["COMP_CLASS"] * w_class
df_calc["SCORE_DISTANCE"] = df_calc["COMP_DISTANCE"] * w_distance
df_calc["SCORE_DRIVER"] = df_calc["COMP_DRIVER"] * w_driver
df_calc["SCORE_TRAINER"] = df_calc["COMP_TRAINER"] * w_trainer
df_calc["SCORE_EARLY"] = df_calc["COMP_EARLY"] * w_early
df_calc["SCORE_FINISH"] = df_calc["COMP_FINISH"] * w_finish
df_calc["SCORE_RECENCY"] = df_calc["COMP_RECENCY"] * w_recency
df_calc["SCORE_COURSE"] = df_calc["COMP_COURSE"] * w_course
df_calc["SCORE_MARKET"] = df_calc["COMP_MARKET"] * w_market

weighted_score_cols = [
    "SCORE_SPEED",
    "SCORE_POWER",
    "SCORE_CLASS",
    "SCORE_DISTANCE",
    "SCORE_DRIVER",
    "SCORE_TRAINER",
    "SCORE_EARLY",
    "SCORE_FINISH",
    "SCORE_RECENCY",
    "SCORE_COURSE",
    "SCORE_MARKET",
]
df_calc["BASE_SKILL_SCORE"] = df_calc[weighted_score_cols].sum(axis=1)

df_calc["FINAL_SCORE"] = round(
    df_calc["BASE_SKILL_SCORE"]
    + df_calc.get("POST_BONUS", 0.0)
    + df_calc.get("STYLE_BONUS", 0.0),
    2,
)

df_calc["RANK"] = (
    df_calc["FINAL_SCORE"].rank(ascending=False, method="min").astype(int)
)
df_calc.sort_values(by="RANK", inplace=True)

# --------------------------------------------------------------------
# DISPLAY FINAL MODEL RANKINGS
# --------------------------------------------------------------------
clean_track_name = clean_track if "clean_track" in locals() else "Track"
clean_race_num = race_num if "race_num" in locals() else "1"

print("=" * 75)
print(f"FINAL MODEL RANKINGS (V16) | {clean_track_name} - RACE #{clean_race_num}")
print("=" * 75)

display_fields = [
    c
    for c in [
        "RANK",
        "PROGRAM",
        "HORSE_NAME",
        "RUN_STYLE",
        "BASE_SKILL_SCORE",
        "POST_BONUS",
        "STYLE_BONUS",
        "FINAL_SCORE",
        "POST",
    ]
    if c in df_calc.columns
]
print(df_calc[display_fields].to_string(index=False))

##------------
# temporary section: only for use in evaluating the model's performance against brisnet prime power top picks
brisnet_cols = ["PROGRAM", "HORSE_NAME", "PRM_PWR"]

if "PRM_PWR" in df_calc.columns:
  df_raw_power = df_calc[brisnet_cols].sort_values(
      by="PRM_PWR", ascending=False
  )
  print("\n--- BRISNET RAW PRIME POWER RANKINGS ---")
  print(df_raw_power.to_string(index=False))
else:
  print("\nPRM_PWR column not found in DataFrame.")
##------------

# -------------------------------------------------------------------------------------------------------------------------------
# BLOCK 5: Wager Recommendations (V16 Synchronized)

if "df_calc" in locals() and len(df_calc) >= 3:
  top_field = df_calc.head(5).copy().reset_index(drop=True)
  num_horses = len(top_field)
  total_starters = len(df_calc)
  p_col = "PROGRAM" if "PROGRAM" in top_field.columns else "POST"

  top_nums = top_field[p_col].astype(str).tolist()
  top_scores = top_field["FINAL_SCORE"].tolist()

  # Pad arrays if fewer than 5 horses in race
  while len(top_nums) < 5:
    top_nums.append("N/A")
    top_scores.append(0.0)

  s1, s2, s3, s4, s5 = top_scores[:5]

  # Calculate Score Gaps across top 5
  gap_12 = round(s1 - s2, 2)
  gap_23 = round(s2 - s3, 2)
  gap_34 = round(s3 - s4, 2)
  gap_13 = round(s1 - s3, 2)
  gap_14 = round(s1 - s4, 2)

  # --------------------------------------------------------------------
  # 1. CHURCHILL DOWNS ODD/EVEN SPECIAL BET CHECK
  # --------------------------------------------------------------------
  churchill_odd_even_rec = None

  if clean_track == "Churchill Downs" and total_starters >= 6:
    all_program_nums = []
    for p in df_calc[p_col]:
      try:
        clean_p = int("".join(filter(str.isdigit, str(p))))
        all_program_nums.append(clean_p)
      except ValueError:
        continue

    odd_count = sum(1 for n in all_program_nums if n % 2 != 0)
    even_count = sum(1 for n in all_program_nums if n % 2 == 0)

    # V16 Threshold Check: Triggered on tight top groups (Gap 1-3 < 4.0 or Gap 1-4 < 5.0)
    if odd_count >= 3 and even_count >= 3 and (gap_13 < 4.0 or gap_14 < 5.0):
      odd_score_sum = 0.0
      even_score_sum = 0.0

      for i in range(min(5, num_horses)):
        try:
          num_val = int("".join(filter(str.isdigit, str(top_nums[i]))))
          if num_val % 2 != 0:
            odd_score_sum += top_scores[i]
          else:
            even_score_sum += top_scores[i]
        except ValueError:
          continue

      preferred_side = "ODD" if odd_score_sum >= even_score_sum else "EVEN"
      side_runners = [
          f"#{top_nums[i]}"
          for i in range(min(5, num_horses))
          if top_nums[i] != "N/A"
          and (
              (int("".join(filter(str.isdigit, str(top_nums[i])))) % 2 != 0)
              if preferred_side == "ODD"
              else (
                  int("".join(filter(str.isdigit, str(top_nums[i])))) % 2 == 0
              )
          )
      ]

      churchill_odd_even_rec = (
          f"CHURCHILL ODD/EVEN WAGER: Bet [{preferred_side}]\n"
          f"      Reason: Contested top group favors {preferred_side} runners"
          f" ({', '.join(side_runners)}) with a score weight of"
          f" {max(odd_score_sum, even_score_sum):.2f} vs"
          f" {min(odd_score_sum, even_score_sum):.2f}."
      )

  # --------------------------------------------------------------------
  # 2. SCENARIO IDENTIFICATION & WAGER FORMULATION (V16 THRESHOLDS)
  # --------------------------------------------------------------------

  # Chaos / Ultra-Tight Field -> 3-Horse Exacta Box (Expanded from 3.0 to 5.0 pts)
  if gap_14 < 5.0 and num_horses >= 4:
    scenario = "Ultra-Tight Field / High Chaos (Top 4 within 5.0 pts)"
    option_a = (
        "PASS / NO BET (Or Value WIN Wager on highest live odds among"
        f" #{top_nums[0]}, #{top_nums[1]}, #{top_nums[2]})"
    )
    option_b = (
        f"3-Horse Exacta Box: #{top_nums[0]}, #{top_nums[1]}, #{top_nums[2]} (6"
        " combos / low cost)"
    )

  # Clear Dominant Winner (Expanded from 5.0 to 8.0 pts)
  elif gap_12 >= 8.0:
    scenario = f"Dominant Standout (#{top_nums[0]} holds >=8.0 pt lead)"
    option_a = f"WIN Wager on #{top_nums[0]}"
    if gap_23 < 4.0 and gap_34 >= 4.0:
      option_b = (
          f"Straight Exacta: #{top_nums[0]} / #{top_nums[1]}, #{top_nums[2]}"
      )
    else:
      option_b = (
          f"Straight Exacta: #{top_nums[0]} / #{top_nums[1]} (Or Exacta Key:"
          f" #{top_nums[0]} / #{top_nums[1]}, #{top_nums[2]}, #{top_nums[3]})"
      )

  # Competitive Top Duo (Gap 1-2 < 8.0 AND Gap 2-3 >= 4.0)
  elif gap_12 < 8.0 and gap_23 >= 4.0:
    scenario = (
        f"Competitive Top Duo (#{top_nums[0]} & #{top_nums[1]} separated from"
        " field)"
    )
    option_a = (
        f"WIN Wager on #{top_nums[0]} (Or PLACE Wager on #{top_nums[1]})"
    )
    option_b = f"Exacta Box: #{top_nums[0]}, #{top_nums[1]}"

  # Volatile Top Trio (Expanded from 3.0 to 4.0 pts)
  elif gap_13 < 4.0:
    scenario = (
        "Volatile Top Group (Top 3 within 4.0 pts:"
        f" #{top_nums[0]}, #{top_nums[1]}, #{top_nums[2]})"
    )
    option_a = (
        "PLACE / SHOW Wager on highest live odds among"
        f" #{top_nums[0]}, #{top_nums[1]}, #{top_nums[2]}"
    )
    option_b = (
        f"Trifecta Box: #{top_nums[0]}, #{top_nums[1]}, #{top_nums[2]} (6"
        " combos / $3.00 total at $0.50 base)"
    )

  # Standard Competitive Field
  else:
    scenario = "Standard Competitive Field"
    option_a = f"WIN / PLACE Wager on #{top_nums[0]}"
    option_b = (
        f"Straight Exacta Wheel: #{top_nums[0]} / #{top_nums[1]}, #{top_nums[2]}"
    )

  # --------------------------------------------------------------------
  # DASHBOARD DISPLAY
  # --------------------------------------------------------------------
  print("\n" + "=" * 70)
  print("MODEL WAGER RECOMMENDATION DASHBOARD (V16)")
  print("=" * 70)
  print(f"Race Scenario Identified: {scenario}")
  print("-" * 70)
  print("Top 5 Field Distribution & Gap Analysis:")
  for i in range(min(5, num_horses)):
    print(
        f"   Rank {i+1}: #{top_nums[i]} - Score: {top_scores[i]:.2f}"
        + (
            f" (Gap to #1: -{s1 - top_scores[i]:.2f})"
            if i > 0
            else " (Leader)"
        )
    )
  print("-" * 70)
  print("OPTION A (Straight Wager - Single Horse Focus):")
  print(f"   -> {option_a}\n")
  print("OPTION B (Exotic Wager - High Payout Alternative):")
  print(f"   -> {option_b}")

  if churchill_odd_even_rec:
    print("\nOPTION C (Special Track Option - Churchill Downs):")
    print(f"   -> {churchill_odd_even_rec}")

  print("=" * 70)
else:
  print(
      "Error: `df_calc` not found or insufficient horses to generate wager"
      " recommendations."
  )

PAUSE 1: Copy your RUNNER CSV to clipboard, then press Enter... 1,"Gdoubleyou",1,6/1,9/5,NA,4,109.1,0,0,71,106.8,106.8,0,0,0.1384,0.1639,0,0,0,19 3,"Jr Bartholomew",3,4/1,3,P,2,105.1,68,76,76,107.1,108.6,68,76,0.0974,0.1477,66,59,83,19 5,"Positive Equity",5,4/1,8,E/P,8,106.5,67,0,57,107.2,104.6,67,0,0.1411,0.1408,62,55,82,10 6,"Gentleman Jim",6,10/1,5/2,P,0,110.7,74,76,72,107.2,107.1,74,76,0.2355,0.0814,70,72,84,19 7,"Night Watcher",7,8/1,5/2,E,8,119.1,71,72,72,106.8,107.1,71,72,0.1449,0.5,83,80,70,19


INGESTING MAIN RUNNER CSV FROM CLIPBOARD
[Auto-Fix] Header-less CSV detected. Applied default schema headers.

Successfully ingested 5 horses across 23 columns!



PAUSE 2: Copy your RACE STATS CSV to clipboard, then press Enter... Thoroughbred,"Horseshoe Indianapolis",6,Meet,78,0.359,1.44,1.08,1.25,0.5,0.99,0.99,1.24,0.65 Thoroughbred,"Horseshoe Indianapolis",6,Week,7,0.1429,2.44,0.81,1.52,0,0,1.06,2.5,0.49


INGESTING RACE STATS CSV FROM CLIPBOARD
[Auto-Fix] Header-less CSV detected. Applied default schema headers.

[Rule Applied] Using 100% Meet stats (Week sample too small: 7 < 15 races)
Active Blended Speed Bias: 0.359
Blended Post Impact Value Map: {'RAIL': 1.44, '1-3': 1.08, '4-7': 1.25, '8+': 0.5}
Blended Run Style Impact Value Map: {'E': 0.99, 'E/P': 0.99, 'P': 1.24, 'S': 0.65}

CALCULATING DYNAMIC POST & STYLE BIAS BONUSES
FINAL MODEL RANKINGS (V16) | Horseshoe Indianapolis - RACE #6
 RANK  PROGRAM      HORSE_NAME RUN_STYLE  BASE_SKILL_SCORE  POST_BONUS  STYLE_BONUS  FINAL_SCORE  POST
    1        7   Night Watcher         E         68.009837        1.50         0.00        69.51     7
    2        6   Gentleman Jim         P         67.781157        1.50         0.00        69.28     6
    3        1      Gdoubleyou       NaN         66.041067        2.64         0.00        68.68     1
    4        3  Jr Bartholomew         P         64.636706        0.48         0.36        65.4